In [1]:
DEEPSEEK_API_KEY="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"
DEEPSEEK_MODEL="deepseek-ai/deepseek-v4-flash-0731"
BASE_URL="https://integrate.api.nvidia.com/v1"

In [2]:
# !pip install langgraph langchain-core langchain-openai langchain-community

In [3]:
# pip install -U ddgs

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from typing import TypedDict
from langgraph.graph import StateGraph , START , END
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_community.tools import DuckDuckGoSearchResults , DuckDuckGoSearchRun

c:\Users\Anirban\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_core\utils\pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1
C:\Users\Anirban\AppData\Local\Temp\ipykernel_19856\4106230801.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchResults , DuckDuckGoSearchRun


In [5]:
def deepseek_llm( temp:float = 0.1)->ChatOpenAI:
    api_key = DEEPSEEK_API_KEY
    if not api_key :
        print("no api key is there ")
    return ChatOpenAI(
        base_url=BASE_URL , 
        api_key=api_key,
        model_name=DEEPSEEK_MODEL,
        temperature=temp,
        streaming=True,
        
    )

In [ ]:
class MovieState(TypedDict):
    movie_query :        str
    raw_web_data:        str
    final_summary :      str
    rating:              str
    is_relevant:         str 

In [7]:
def fetch_movie_data(state:MovieState):
    query= state["movie_query"]
    search_tool= DuckDuckGoSearchRun()

    search_results  = search_tool.invoke(f"{query} movie plot and reception")
    return {
        "raw_web_data":search_results
    }

In [8]:
def summarize_data(state:MovieState):
    raw_data = state["raw_web_data"]
    llm = deepseek_llm()
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful movie expert. Summarize the following raw web search data into a concise, 3-sentence movie summary covering the plot and general reception."),
        ("human", "{data}")
    ])

    chain = prompt | llm
    response = chain.invoke({
        "data":raw_data
    })

    return {
        "final_summary":response.content
    }

In [9]:
def grade_documents(state: MovieState):
    raw_data = state["raw_web_data"]
    query = state["movie_query"]
    
    # We use temperature=0 for grading to get deterministic yes/no answers
    llm = deepseek_llm(temp=0.0) 
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a grader assessing the relevance of a retrieved document. "
                   "If the document contains information about the requested movie, grade it as 'yes'. Otherwise 'no'. "
                   "Only answer with the word 'yes' or 'no' and absolutely nothing else."),
        ("human", f"Movie Requested: {query}\n\nRetrieved Data: {raw_data}")
    ])
    
    chain = prompt | llm
    response = chain.invoke({})
    grade = response.content.strip().lower()
    
    return {"is_relevant": grade}


In [10]:
def rewrite_query(state: MovieState):
    query = state["movie_query"]
    llm = deepseek_llm(temp=0.7)
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an expert at writing search engine queries. The previous search failed to find the movie. "
                   "Write a better search query to find the plot and reception of the movie. "
                   "Just output the new query string, nothing else."),
        ("human", f"Original Query: {query}")
    ])
    
    chain = prompt | llm
    response = chain.invoke({})
    new_query = response.content.strip()
    
    print(f"--- REWRITING QUERY TO: {new_query} ---")
    return {"movie_query": new_query}


In [11]:
def decide_to_generate(state: MovieState):
    if state["is_relevant"] == "yes":
        print("--- DECISION: DATA IS RELEVANT, PROCEED TO SUMMARIZE ---")
        return "generate"
    else:
        print("--- DECISION: DATA IS IRRELEVANT, REWRITE QUERY ---")
        return "rewrite"


In [12]:
workflow = StateGraph(MovieState)

workflow.add_node("fetch", fetch_movie_data)
workflow.add_node("grade", grade_documents)
workflow.add_node("rewrite", rewrite_query)
workflow.add_node("summarize", summarize_data)

workflow.add_edge(START, "fetch")      
workflow.add_edge("fetch", "grade") 
workflow.add_conditional_edges(
    "grade",
    decide_to_generate,
    {
        "generate": "summarize", 
        "rewrite": "rewrite"    
    }
)
workflow.add_edge("rewrite", "fetch") 
workflow.add_edge("summarize", END)

app = workflow.compile()


In [13]:
MOVIE_NAME = "Mirzapur"
if __name__ == "__main__":
    initial_state = {
        "movie_query" : MOVIE_NAME
    }
    result = app.invoke(initial_state)
    print("\n--- FINAL OUTPUT ---")
    print(result["final_summary"])
    # with open("raw_movie_data.txt", "w", encoding="utf-8") as file:
    #     file.write(result["raw_web_data"])

--- DECISION: DATA IS RELEVANT, PROCEED TO SUMMARIZE ---

--- FINAL OUTPUT ---
Set in 2018, the film follows Akhandanand "Kaleen" Tripathi as he continues to rule the violent underworld of Mirzapur, where a power struggle escalates as old enemies resurface and new contenders fight for control. The movie offers a comforting familiarity and thrill of surprise by retaining the show's core cast and introducing new arcs, but it loses its grip by prioritizing the series' complex, layered storylines over a cohesive cinematic narrative. Critics note that with a runtime of 3 hours and 17 minutes, the makers forget this is a film, not a 10-episode season, leading to a bloated and unfocused experience.
